# Análisis Geoespacial de Calidad del Aire
## Posadas, Misiones, Argentina

Este notebook realiza análisis geoespacial de los datos de calidad del aire, incluyendo:
- Mapas interactivos con Folium
- Distribución espacial de contaminantes
- Análisis de proximidad
- Mapas de calor geográficos

In [ ]:
# Importar librerías
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap, MarkerCluster
import warnings
warnings.filterwarnings('ignore')

from config.config import LOCATION, BBOX, POLLUTANTS, AQI_CATEGORIES

print("✓ Librerías cargadas correctamente")

## 1. Cargar Datos

In [ ]:
# Cargar datos procesados
df = pd.read_csv('../data/processed/processed_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

print(f"📊 Datos cargados: {len(df)} registros")
print(f"📍 Ubicaciones únicas: {df['location'].nunique()}")
print(f"🏷️ Contaminantes: {df['parameter'].unique()}")

df.head()

## 2. Mapa Base de Posadas

In [ ]:
# Crear mapa centrado en Posadas
posadas_map = folium.Map(
    location=[LOCATION['latitude'], LOCATION['longitude']],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Agregar marcador de la ciudad
folium.Marker(
    location=[LOCATION['latitude'], LOCATION['longitude']],
    popup=f"<b>{LOCATION['name']}</b><br>{LOCATION['province']}, {LOCATION['country']}",
    tooltip='Ciudad de Posadas',
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(posadas_map)

# Agregar área de estudio (bounding box)
folium.Rectangle(
    bounds=[
        [BBOX['min_lat'], BBOX['min_lon']],
        [BBOX['max_lat'], BBOX['max_lon']]
    ],
    color='blue',
    fill=True,
    fill_opacity=0.1,
    popup='Área de estudio (50km radio)'
).add_to(posadas_map)

# Guardar mapa
posadas_map.save('../output/figures/mapa_base_posadas.html')
print("💾 Mapa guardado en output/figures/mapa_base_posadas.html")

posadas_map

## 3. Mapa de Estaciones de Monitoreo

In [ ]:
# Obtener ubicaciones únicas de estaciones
stations = df[['location', 'latitude', 'longitude']].drop_duplicates()

# Crear mapa
stations_map = folium.Map(
    location=[LOCATION['latitude'], LOCATION['longitude']],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Agregar marcadores para cada estación
for idx, station in stations.iterrows():
    # Calcular estadísticas de la estación
    station_data = df[df['location'] == station['location']]
    n_measurements = len(station_data)
    avg_aqi = station_data['aqi'].mean() if 'aqi' in station_data.columns else 0
    
    # Color basado en AQI promedio
    if avg_aqi <= 50:
        color = 'green'
    elif avg_aqi <= 100:
        color = 'orange'
    elif avg_aqi <= 150:
        color = 'red'
    else:
        color = 'purple'
    
    # Popup con información
    popup_html = f"""
    <div style="font-family: Arial; width: 200px;">
        <h4>{station['location']}</h4>
        <b>Mediciones:</b> {n_measurements}<br>
        <b>AQI Promedio:</b> {avg_aqi:.1f}<br>
        <b>Coordenadas:</b> {station['latitude']:.4f}, {station['longitude']:.4f}
    </div>
    """
    
    folium.Marker(
        location=[station['latitude'], station['longitude']],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=station['location'],
        icon=folium.Icon(color=color, icon='stats', prefix='fa')
    ).add_to(stations_map)

# Guardar mapa
stations_map.save('../output/figures/mapa_estaciones.html')
print(f"💾 Mapa de estaciones guardado")
print(f"📍 Total de estaciones: {len(stations)}")

stations_map

## 4. Mapa de Calor - PM2.5

In [ ]:
# Filtrar datos de PM2.5
pm25_data = df[df['parameter'] == 'pm25'].copy()

# Calcular promedio por ubicación
pm25_avg = pm25_data.groupby(['latitude', 'longitude']).agg({
    'value': 'mean',
    'location': 'first'
}).reset_index()

# Crear mapa de calor
heatmap_pm25 = folium.Map(
    location=[LOCATION['latitude'], LOCATION['longitude']],
    zoom_start=12,
    tiles='CartoDB positron'
)

# Preparar datos para HeatMap
heat_data = [[row['latitude'], row['longitude'], row['value']] 
             for idx, row in pm25_avg.iterrows()]

# Agregar capa de calor
HeatMap(
    heat_data,
    min_opacity=0.3,
    max_val=pm25_avg['value'].max(),
    radius=25,
    blur=35,
    gradient={
        0.0: 'green',
        0.3: 'yellow',
        0.5: 'orange',
        0.7: 'red',
        1.0: 'purple'
    }
).add_to(heatmap_pm25)

# Agregar leyenda
legend_html = f'''
<div style="position: fixed; 
            top: 10px; right: 10px; width: 200px; height: 140px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<h4 style="margin-top:0;">PM2.5 (μg/m³)</h4>
<p><b>Mínimo:</b> {pm25_avg['value'].min():.1f}</p>
<p><b>Promedio:</b> {pm25_avg['value'].mean():.1f}</p>
<p><b>Máximo:</b> {pm25_avg['value'].max():.1f}</p>
<p style="color:red;"><b>Guía OMS:</b> {POLLUTANTS['pm25']['who_guideline_24h']}</p>
</div>
'''
heatmap_pm25.get_root().html.add_child(folium.Element(legend_html))

# Guardar
heatmap_pm25.save('../output/figures/mapa_calor_pm25.html')
print("💾 Mapa de calor PM2.5 guardado")

heatmap_pm25

## 5. Mapa Interactivo con Múltiples Contaminantes

In [ ]:
# Crear mapa con capas
multi_map = folium.Map(
    location=[LOCATION['latitude'], LOCATION['longitude']],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Crear grupos de capas para cada contaminante
pollutants_list = ['pm25', 'pm10', 'no2', 'o3']

for pollutant in pollutants_list:
    # Filtrar datos
    poll_data = df[df['parameter'] == pollutant].copy()
    
    if poll_data.empty:
        continue
    
    # Promediar por ubicación
    poll_avg = poll_data.groupby(['latitude', 'longitude']).agg({
        'value': 'mean',
        'location': 'first'
    }).reset_index()
    
    # Crear feature group
    feature_group = folium.FeatureGroup(name=f'{POLLUTANTS[pollutant]["name"]} - {POLLUTANTS[pollutant]["full_name"]}')
    
    # Agregar marcadores
    for idx, row in poll_avg.iterrows():
        # Determinar color según valor
        value = row['value']
        guideline = POLLUTANTS[pollutant].get('who_guideline_24h', float('inf'))
        
        if value <= guideline * 0.5:
            color = 'green'
        elif value <= guideline:
            color = 'orange'
        else:
            color = 'red'
        
        # Popup
        popup_html = f"""
        <div style="font-family: Arial;">
            <h4>{POLLUTANTS[pollutant]['name']}</h4>
            <b>Ubicación:</b> {row['location']}<br>
            <b>Promedio:</b> {value:.1f} {POLLUTANTS[pollutant]['unit']}<br>
            <b>Guía OMS:</b> {guideline} {POLLUTANTS[pollutant]['unit']}
        </div>
        """
        
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=10,
            popup=folium.Popup(popup_html, max_width=250),
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.6
        ).add_to(feature_group)
    
    feature_group.add_to(multi_map)

# Agregar control de capas
folium.LayerControl().add_to(multi_map)

# Guardar
multi_map.save('../output/figures/mapa_interactivo_multiple.html')
print("💾 Mapa interactivo guardado")

multi_map

## 6. Análisis de Distribución Espacial

In [ ]:
# Crear gráfico de dispersión geográfica
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

pollutants_to_plot = ['pm25', 'pm10', 'no2', 'o3']

for idx, pollutant in enumerate(pollutants_to_plot):
    ax = axes[idx]
    
    poll_data = df[df['parameter'] == pollutant]
    
    if not poll_data.empty:
        # Scatter plot
        scatter = ax.scatter(
            poll_data['longitude'],
            poll_data['latitude'],
            c=poll_data['value'],
            cmap='YlOrRd',
            s=50,
            alpha=0.6,
            edgecolors='black',
            linewidth=0.5
        )
        
        # Colorbar
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label(f'{POLLUTANTS[pollutant]["unit"]}', fontsize=10)
        
        # Centro de Posadas
        ax.plot(LOCATION['longitude'], LOCATION['latitude'], 
               'r*', markersize=20, label='Posadas Centro')
        
        ax.set_xlabel('Longitud', fontsize=11)
        ax.set_ylabel('Latitud', fontsize=11)
        ax.set_title(f'{POLLUTANTS[pollutant]["name"]} - Distribución Espacial',
                    fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle(f'Distribución Geográfica de Contaminantes - {LOCATION["name"]}',
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()

# Guardar
plt.savefig('../output/figures/distribucion_espacial.png', dpi=300, bbox_inches='tight')
print("💾 Gráfico de distribución espacial guardado")

plt.show()

## 7. Resumen Geoespacial

In [ ]:
print("="*70)
print("RESUMEN DE ANÁLISIS GEOESPACIAL")
print("="*70)
print(f"\nÁrea de estudio: {LOCATION['name']}, {LOCATION['province']}")
print(f"Coordenadas centrales: {LOCATION['latitude']}, {LOCATION['longitude']}")
print(f"Radio de cobertura: ~50 km")
print(f"\nEstaciones de monitoreo: {df['location'].nunique()}")
print(f"Total de mediciones: {len(df)}")
print(f"\nMapas generados:")
print("  1. Mapa base de Posadas")
print("  2. Mapa de estaciones de monitoreo")
print("  3. Mapa de calor PM2.5")
print("  4. Mapa interactivo multi-contaminante")
print("  5. Distribución espacial de contaminantes")
print("\n✓ Análisis geoespacial completado")
print("="*70)